# ***Task 1.1 — Core Contribution / Architecture***

**Paper:** Pegasos: Primal Estimated sub-GrAdient SOlver for SVM  
**Authors:** Shai Shalev-Shwartz, Yoram Singer, Nathan Srebro, Andrew Cotter  
**Published in:** Mathematical Programming, 2011 (Original ICML 2007)

## *Step-by-Step Method Description*

### *Step 1: Problem Formulation as a Primal Objective*
- **Description:** The method begins by expressing binary linear SVM training as the minimization of a regularised hinge-loss over all training examples. The weight vector `w` is sought, not the dual variables `α`.
- **Reference:** Equation (1) in Section 2 of the paper.
   $$\min_{w} \; f(w) = \frac{\lambda}{2}\|w\|^2 + \frac{1}{m}\sum_{(x,y)\in S} \max(0, 1 - y\langle w, x\rangle)$$
- **Purpose:** By working in the primal space rather than the dual, the algorithm avoids the need to store or compute an `m×m` kernel matrix, making it feasible for very large datasets.

### *Step 2: Stochastic Mini-Batch Selection*
- **Description:** At each iteration `t`, a random mini-batch `A_t` of size `k` is sampled uniformly at random (with replacement) from the training set `S`. A typical choice is `k = 1` (pure stochastic gradient).
- **Reference:** Algorithm 1 in Section 2 / Mini-batch extension in Section 3.1.
- **Purpose:** Using a mini-batch instead of the full dataset drastically reduces the cost per iteration from `O(md)` to `O(kd)`, enabling fast convergence in wall-clock time even for very large `m`.

### *Step 3: Identify Margin-Violating Examples*
- **Description:** Within the sampled mini-batch `A_t`, only the subset `A_t⁺` of examples that actually violate the margin (i.e., those for which `y⟨w_t, x⟩ < 1`) contribute to the subgradient. Examples already correctly classified with large margin are ignored.
- **Reference:** Definition of `A_t⁺` below Algorithm 1 in Section 2.
- **Purpose:** This corresponds to taking the subgradient of the hinge loss `max(0, 1 − y⟨w,x⟩)`, which is 0 outside the margin and `−yx` inside it. This mirrors the SVM support-vector structure: only support vectors matter.

### *Step 4: Learning Rate Schedule*
- **Description:** The learning rate (step size) for iteration `t` is set to `η_t = 1/(λt)`. It starts relatively large and decreases as the number of iterations grows, following a `1/t` decay schedule.
- **Reference:** Equation (2) / Theorem 1 (Section 2) — the specific choice of `η_t = 1/(λt)` is critical to the theoretical guarantee.
- **Purpose:** The `1/(λt)` schedule is precisely calibrated so that the algorithm converges at a rate of `O(1/(λε))` iterations to reach ε-accuracy, **independent of m**. This is the key theoretical insight of the paper.

### *Step 5: Sub-Gradient Update Step*
- **Description:** The weight vector is updated by first scaling it by `(1 − η_t λ) = (1 − 1/t)` (a shrinkage step implementing the L2 regulariser) and then adding the contribution from the margin-violating examples:
   $$w_{t+1/2} = \left(1 - \frac{1}{t}\right)w_t + \frac{1}{\lambda t k}\sum_{(x,y)\in A_t^{+}} yx$$
- **Reference:** Equations (2)–(3) in Section 2, Algorithm 1.
- **Purpose:** The first term is a multiplicative weight decay (regularisation), and the second term is the standard perceptron-style update for misclassified examples. Together they minimise the primal SVM objective.

### *Step 6: Projection onto the L2 Ball (Optional Projection Step)*
- **Description:** After the gradient step, the intermediate weight vector `w_{t+1/2}` is optionally projected back onto the Euclidean ball of radius `1/√λ`. This is done by the operation:
   $$w_{t+1} = \min\left\{1, \frac{1/\sqrt{\lambda}}{\|w_{t+1/2}\|}\right\} w_{t+1/2}$$
- **Reference:** Equation (4) in Section 2.
- **Purpose:** The optimal SVM solution `w*` is guaranteed to lie within this ball. Projecting ensures iterates never stray too far from the feasible region, which tightens the theoretical convergence guarantee (though in practice the projection rarely activates).

### *Step 7: Iterate and Return Final Weight Vector*
- **Description:** Steps 2–6 are repeated for `T` iterations, where `T = O(1/(λε))`. At the end, the algorithm returns the **last iterate** `w_T` as the final SVM weight vector.
- **Reference:** Theorem 1, Section 2.
- **Purpose:** The theoretical analysis proves that the last iterate `w_T` with high probability achieves objective value within ε of the optimal, establishing Pegasos as a convergent solver.

## ***Final Summary***

This paper solves the large-scale linear SVM training problem by proposing a stochastic primal sub-gradient method (Pegasos) that converges in **O(1/(λε)) iterations regardless of the dataset size m**, which the authors claim makes it orders-of-magnitude faster than existing dual-based solvers such as SVM-Light and SVM-Perf that scale with m.
